In [1]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "travel_server": {
            "transport": "streamable_http",
            "url": "https://mcp.kiwi.com"
        }
    }
)

tools = await client.get_tools()

In [2]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    "gpt-5-nano",
    tools=tools,
    checkpointer=InMemorySaver(),
    system_prompt="You're a helpful travel assistant. No follow-up questions.",
)

In [3]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Get me a direct flight from San Francisco to Tokyo on March 31st")]},
    config
)

In [4]:
for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

Get me a direct flight from San Francisco to Tokyo on March 31st
================================== Ai Message ==================================
Tool Calls:
  search-flight (call_yloWfNCR4ZPHnH9mrtACo51h)
 Call ID: call_yloWfNCR4ZPHnH9mrtACo51h
  Args:
    flyFrom: SFO
    flyTo: Tokyo
    departureDate: 31/03/2026
================================= Tool Message =================================
Name: search-flight

[{'type': 'text', 'text': '[\n  {\n    "flyFrom": "SFO",\n    "flyTo": "NRT",\n    "cityFrom": "San Francisco",\n    "cityTo": "Tokyo",\n    "departure": {\n      "utc": "2026-03-31T07:50:00.000Z",\n      "local": "2026-03-31T00:50:00.000"\n    },\n    "arrival": {\n      "utc": "2026-04-01T03:55:00.000Z",\n      "local": "2026-04-01T12:55:00.000"\n    },\n    "totalDurationInSeconds": 72300,\n    "durationInSeconds": 72300,\n    "price": 442,\n    "deepLink": "https://on.kiwi.com/GO7tH6",\n  

In [5]:
print(response["messages"][-1].content)

Here’s the direct flight I found for March 31 from San Francisco to Tokyo:

| Route | Schedule (local times) | Cabin | Price | Booking link |
|---|---|---|---|---|
| SFO → NRT | 31/03 16:45 → 01/04 20:00 (11h 15m) | Economy | 329 EUR | https://on.kiwi.com/7Y8jWN |

Summary:
- Best price: 329 EUR
- Shortest direct flight: 11h 15m
- Recommendation: Direct overnight SFO to NRT on March 31

Fun fact: Tokyo is home to the busiest pedestrian crossing in the world at Shibuya Crossing, where hundreds of people cross in every light change. Have a fantastic trip!
